<a href="https://colab.research.google.com/github/mompaduck/Data-Analysis-with-Open-Source/blob/main/%EC%98%A4%ED%94%88%EC%86%8C%EC%8A%A4_%EB%8D%B0%EC%9D%B4%ED%84%B0_%EB%B6%84%EC%84%9D_%EC%A4%91%EA%B0%84%EA%B3%BC%EC%A0%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf
#한글깨짐 방지 -폰트설치 코드



In [ ]:

#한국농수산식품유통공사 일별 도,소매 가격정보 조회의 데이터를 Open API를 활용하여 수집

import requests
import pandas as pd

api_key ='f475cd43dd3ea7f358af7c75801ceaa4d18c495f631d171dd2b8c16e607daccd'

url = 'https://apis.data.go.kr/B552845/perDay/price'

# 1. API 데이터 요청 함수
def request_data(year, variety_code, page_no=1, num_of_rows=200):

    params = {
        'serviceKey': api_key,
        'pageNo': page_no,
        'numOfRows': num_of_rows,
        'returnType': 'JSON',

        'cond[exmn_ymd::GTE]': f'{year}0101',
        'cond[exmn_ymd::LTE]': f'{year}1231',

        'cond[se_cd::EQ]': '02',
        'cond[ctgry_cd::EQ]': '200',
        'cond[item_cd::EQ]': '211',
        'cond[grd_cd::EQ]': '04',
        'cond[sgg_cd::EQ]': '1101',
        'cond[mrkt_cd::EQ]': '0110211',
        'cond[vrty_cd::EQ]': variety_code
    }

    try:
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()

        # 성공 로그 (연도, 품종, 페이지 번호 명시)
        print(f'[{year}년] [품종코드:{variety_code}] 🟢 API 호출 성공 (HTTP {response.status_code})')
        return response.json()


    except requests.exceptions.Timeout:
        print(f'[{year}년] [품종코드:{variety_code}] ⏱️ 요청 시간 초과')
        return None

    except requests.exceptions.RequestException as e:
        print(f'[{year}년] [품종코드:{variety_code}] ❌ API 요청 오류: {e}')
        return None

    except ValueError:
        print(f'[{year}년] [품종코드:{variety_code}] ❌ JSON 변환 오류')
        return None




# 배추 품종 코드
varieties = {
    '01': '봄배추',
    '02': '여름배추(고랭지)',
    '03': '가을배추',
    '06': '월동배추'
}

# 수집 기간 설정
start_year = 2015
end_year = 2024


# 전체 데이터 저장 리스트
all_data = []

# 2. 데이터 수집 루프
print('=== API 데이터 수집 시작 ===')
for year in range(start_year, end_year + 1):
    for variety_code, variety_name in varieties.items():

        data = request_data(year, variety_code)

        # API 응답 실패 처리
        if not data:
            continue
        try:
            body = data.get('response', {}).get('body', {})
            total_count = body.get('totalCount', 0)
            items = body.get('items', [])

            # API 응답 구조 예외 처리 (dict 형태로 들어올 경우 대응)
            if isinstance(items, dict):
                items = items.get('item', [])

            print(f'  └ [{variety_name}] totalCount: {total_count}건')
            all_data.extend(items)

        except Exception as e:
            print(f'  └ [{variety_name}] ❌ 데이터 파싱 오류: {e}')

print('=== API 데이터 수집 완료 ===\n')


In [ ]:
import pandas as pd

# 월별 계절 매핑 딕셔너리
SEASON_MAP = {
    12: '겨울', 1: '겨울', 2: '겨울',
    3: '봄', 4: '봄', 5: '봄',
    6: '여름', 7: '여름', 8: '여름',
    9: '가을', 10: '가을', 11: '가을'
}

# 수집된 리스트 데이터를 DataFrame으로 변환하고 날짜/계절 전처리를 수행하는 함수
def process_data(all_data):

    if not all_data:
        print('❌ 수집된 데이터가 없습니다.')
        return None

    # 1. DataFrame 변환
    df = pd.DataFrame(all_data)

    # 2. 데이터 기본 정보 출력
    print('=== 데이터 기본 정보 ===')
    print('데이터 크기(행, 열):', df.shape)
    print('\n컬럼 목록:', df.columns.tolist())
    print('\n데이터 타입:\n', df.dtypes)
    print('\n데이터 상단 5건:\n', df.head())
    print('\n기술 통계량:\n', df.describe(include='all'))

    # 3. 날짜형 변환 및 연도/계절 컬럼 추가
    if 'exmn_ymd' in df.columns:
        df['exmn_ymd'] = pd.to_datetime(df['exmn_ymd'], format='%Y%m%d')
        df['year'] = df['exmn_ymd'].dt.year
        df['season'] = df['exmn_ymd'].dt.month.map(SEASON_MAP)

        print('\n=== 연도 및 계절 전처리 결과 (상단 20건) ===')
        print(df[['exmn_ymd', 'year', 'season']].head(20))

    return df





# 3. 전처리 함수 호출
df = process_data(all_data)



In [ ]:
import matplotlib.pyplot as plt
plt.rc('font', family='NanumBarunGothic')

In [ ]:
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 연도별 배추 평균 가격 변화량을 선 그래프로 시각화하고 저장하는 함수
def plot_yearly_price(df, student_id_last4="1234"):

    if df is None or df.empty or 'exmn_dd_prc' not in df.columns:
        print('❌ 시각화할 가격 데이터(exmn_dd_prc)가 존재하지 않습니다.')
        return

    # 1. 가격 컬럼 숫자형 변환 (문자열 또는 이상치 예외 처리)
    df['exmn_dd_prc'] = pd.to_numeric(df['exmn_dd_prc'], errors='coerce')

    # 2. 연도별 평균 가격 집계
    yearly_avg = df.groupby('year')['exmn_dd_prc'].mean().reset_index()

    # 3. 한글 폰트 설정
    plt.rc('font', family='NanumBarunGothic')
    plt.rc('axes', unicode_minus=False)

    # 4. 그래프 생성
    plt.figure(figsize=(8, 5))
    plt.plot(
        yearly_avg['year'],
        yearly_avg['exmn_dd_prc'],
        marker='o',
        color='#1f77b4',
        linewidth=1.8,
        markersize=6
    )

  # 각 데이터 점(마커) 위에 구체적인 평균 가격 수치 표시
    for _, row in yearly_avg.iterrows():
        year = row['year']
        price = row['exmn_dd_prc']

        if pd.notna(price):
            plt.text(
                year,                      # X축 위치 (연도)
                price + (price * 0.02),    # Y축 위치 (점보다 약 2% 위에 배치)
                f'{price:,.0f}원',          # 천 단위 쉼표(,) 및 원 단위 포맷팅
                ha='center',
                va='bottom',
                fontsize=9,
                fontweight='bold'
            )

    # 5. 제목 및 축 라벨 설정 (학번 뒤 4자리 포함)
    title_text = f'연도별 배추 평균 가격 변화 - {student_id_last4}'
    plt.title(title_text, fontsize=14, fontweight='bold', pad=15)
    plt.xlabel('연도 (Year)', fontsize=11, labelpad=8)
    plt.ylabel('평균 가격 (원/포장단위)', fontsize=11, labelpad=8)

    # 수치 텍스트가 상단에서 잘리지 않도록 Y축 여백 확보
    max_val = yearly_avg['exmn_dd_prc'].max()
    min_val = yearly_avg['exmn_dd_prc'].min()
    if pd.notna(max_val) and pd.notna(min_val):
        plt.ylim(min_val * 0.85, max_val * 1.12)


    # X축 연도 눈금 설정 (정수 표현)
    plt.xticks(yearly_avg['year'].astype(int))
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()

    # 6. 이미지 저장 및 출력
    filename = f'cabbage_price_line_{student_id_last4}.png'
    plt.savefig(filename, dpi=300)
    print(f'🟢 그래프 저장 완료: {filename}')

    plt.show()

# 계절별 배추 평균 가격을 막대 그래프 - 각 막대 상단에 구체적인 수치를 표시
def plot_seasonal_price(df, student_id_last4="1234"):

    if df is None or df.empty or 'exmn_dd_prc' not in df.columns or 'season' not in df.columns:
        print('❌ 시각화할 데이터(exmn_dd_prc, season)가 부족합니다.')
        return

    # 1. 가격 컬럼 숫자형 변환
    df['exmn_dd_prc'] = pd.to_numeric(df['exmn_dd_prc'], errors='coerce')

    # 2. 계절 정렬 순서 및 평균 가격 집계 (봄 -> 여름 -> 가을 -> 겨울)
    season_order = ['봄', '여름', '가을', '겨울']
    seasonal_avg = df.groupby('season')['exmn_dd_prc'].mean().reindex(season_order).reset_index()

    # 3. 한글 폰트 및 마이너스 기호 설정
    plt.rc('font', family='NanumBarunGothic')
    plt.rc('axes', unicode_minus=False)

    # 4. 막대 그래프 생성
    plt.figure(figsize=(8, 5))
    colors = ['#ff9999', '#66b3ff', '#99ff99', '#ffcc99']  # 계절별 구분 색상

    bars = plt.bar(
        seasonal_avg['season'],
        seasonal_avg['exmn_dd_prc'],
        color=colors,
        edgecolor='gray',
        width=0.55
    )

    # 5. 각 막대 상단에 구체적인 수치(가격) 표시
    for bar in bars:
        height = bar.get_height()
        if pd.notna(height):  # 데이터가 존재하는 경우만 표시
            plt.text(
                bar.get_x() + bar.get_width() / 2.0,  # X축 위치 (막대 중앙)
                height + (height * 0.015),             # Y축 위치 (막대 살짝 위)
                f'{height:,.0f}원',                    # 1,000단위 콤마 포맷팅
                ha='center',
                va='bottom',
                fontsize=10,
                fontweight='bold'
            )

    # 6. 제목 및 축 라벨 설정
    title_text = f'계절별 배추 평균 가격 - {student_id_last4}'
    plt.title(title_text, fontsize=14, fontweight='bold', pad=15)
    plt.xlabel('계절 (Season)', fontsize=11, labelpad=8)
    plt.ylabel('평균 가격 (원/포장단위)', fontsize=11, labelpad=8)

    # Y축 여백 조절 (숫자 텍스트가 잘리지 않도록 상단 여백 확보)
    max_val = seasonal_avg['exmn_dd_prc'].max()
    if pd.notna(max_val):
        plt.ylim(0, max_val * 1.15)

    plt.grid(axis='y', linestyle='--', alpha=0.5)
    plt.tight_layout()

    # 이미지 저장 및 출력
    filename = f'cabbage_price_bar_{student_id_last4}.png'
    plt.savefig(filename, dpi=300)
    print(f'🟢 그래프 저장 완료: {filename}')

    plt.show()


# 4. 연도별 배추 평균 가격(exmn_dd_prc) 변화량 - 선그래프
plot_yearly_price(df)

# 절별 배추 평균 가격을 막대 - 그래프
plot_seasonal_price(df)